# Living Tales — SceneLM v3 Training (Colab)

Two-stage run with live validation: `base` (union universal-dim corpus) → per-case LoRA `adapters`.
Every run writes **directly to Drive**: periodic checkpoints + resumable `train_state.pt` + `train.jsonl`
(loss + val_loss) + `eval.jsonl` (6-probe trend every EVAL_EVERY steps). Safe to disconnect —
re-run the training cells with RESUME on and they continue from the last checkpoint.

Flow: setup → validate data → base → adapters → curves → eval gate → error analysis → transcripts.


In [ ]:
# ── 00 GPU + Drive ──────────────────────────────────────────────────────────
import subprocess
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                     capture_output=True, text=True).stdout or 'NO GPU — Runtime > Change runtime type > T4')
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# ── 01 Repo (uncomment ONE method) ──────────────────────────────────────────
import os, subprocess, zipfile

METHOD         = 'drive'
DRIVE_ZIP_PATH = '/content/drive/MyDrive/more_than_words.zip'   # ← zip of the repo
DRIVE_REPO_DIR = '/content/more_than_words'                     # extract target (local disk = faster)

# METHOD, GITHUB_URL, GITHUB_TOKEN = 'github', 'https://github.com/YOUR_ORG/more_than_words.git', ''

if METHOD == 'drive':
    if not os.path.isdir(DRIVE_REPO_DIR):
        with zipfile.ZipFile(DRIVE_ZIP_PATH) as z:
            z.extractall('/content')
    REPO = DRIVE_REPO_DIR
elif METHOD == 'github':
    url = GITHUB_URL.replace('https://', f'https://{GITHUB_TOKEN}@') if GITHUB_TOKEN else GITHUB_URL
    subprocess.run(['git', 'clone', '--depth', '1', url, '/content/more_than_words'], check=True)
    REPO = '/content/more_than_words'

TRAINER = f'{REPO}/living_tales/trainer'
os.chdir(TRAINER)
print('repo at', REPO)


In [ ]:
# ── 02 Deps + config ────────────────────────────────────────────────────────
%pip -q install -r requirements.txt

OUTPUT_DIR = '/content/drive/MyDrive/living_tales_outputs'   # Drive-safe: checkpoints land here as written
CASES      = ['amber_cipher', 'attended_hour', 'venetian_mirror']
BASE_STEPS, ADAPTER_STEPS = 3000, 1500
CHECKPOINT_EVERY, LOG_EVERY, EVAL_EVERY = 250, 50, 500
import os; os.makedirs(OUTPUT_DIR, exist_ok=True)


In [ ]:
# ── 03 Validate data (must be 0/0 for all three) ────────────────────────────
for c in CASES:
    !PYTHONPATH=. python3 tools/validate_trajectories.py {c} --all | tail -1


In [ ]:
# ── 04 Stage 1: base pretrain (~minutes on T4; resume-safe) ─────────────────
!PYTHONPATH=. python3 tools/train_scene_lm.py base \
  --output-dir {OUTPUT_DIR} --steps {BASE_STEPS} \
  --checkpoint-every {CHECKPOINT_EVERY} --log-every {LOG_EVERY} --resume


In [ ]:
# ── 05 Stage 2: per-case adapters with mid-training 6-probe eval ────────────
for c in CASES:
    print(f'══ adapter: {c} ══')
    !PYTHONPATH=. python3 tools/train_scene_lm.py adapter --case {c} \
      --output-dir {OUTPUT_DIR} --steps {ADAPTER_STEPS} \
      --checkpoint-every {CHECKPOINT_EVERY} --log-every {LOG_EVERY} \
      --eval-every {EVAL_EVERY} --resume


In [ ]:
# ── 06 Loss curves (train + val) ────────────────────────────────────────────
import json, matplotlib.pyplot as plt
from pathlib import Path

dirs = [('base', Path(OUTPUT_DIR) / '_base')] + [(c, Path(OUTPUT_DIR) / c) for c in CASES]
fig, axes = plt.subplots(1, len(dirs), figsize=(5 * len(dirs), 3.5))
for ax, (name, d) in zip(axes, dirs):
    f = d / 'train.jsonl'
    if not f.exists():
        ax.set_title(f'{name} (no log)'); continue
    rows = [json.loads(l) for l in f.read_text().splitlines()]
    ax.plot([r['step'] for r in rows], [r['loss'] for r in rows], label='train')
    vl = [(r['step'], r['val_loss']) for r in rows if r.get('val_loss') is not None]
    if vl: ax.plot(*zip(*vl), label='val')
    ax.set_title(name); ax.legend()
plt.tight_layout(); plt.show()


In [ ]:
# ── 07 Probe trends (are we training in the right direction?) ───────────────
PROBES = ['binding', 'diversity_max', 'outcome_acc', 'conv_spearman']
BARS   = {'binding': 0.95, 'diversity_max': 0.40, 'outcome_acc': 0.90, 'conv_spearman': 0.70}
fig, axes = plt.subplots(1, len(PROBES), figsize=(5 * len(PROBES), 3.5))
for ax, p in zip(axes, PROBES):
    for c in CASES:
        f = Path(OUTPUT_DIR) / c / 'eval.jsonl'
        if not f.exists(): continue
        rows = [json.loads(l) for l in f.read_text().splitlines()]
        ax.plot([r['step'] for r in rows], [r[p] for r in rows], marker='o', label=c)
    ax.axhline(BARS[p], ls='--', c='grey')
    ax.set_title(p); ax.legend()
plt.tight_layout(); plt.show()


In [ ]:
# ── 08 Final eval gate (exit 1 on any probe failure) ────────────────────────
for c in CASES:
    print(f'══ {c} ══')
    !PYTHONPATH=. python3 tools/eval_scene_lm.py {c} \
      --model-path {OUTPUT_DIR}/{c}/scene_lm_full.pt \
      --report {OUTPUT_DIR}/{c}/scene_lm_eval.json | tail -8


In [ ]:
# ── 10 Playtest transcripts → Drive (judge sweep runs locally) ──────────────
import shutil
for c in CASES:
    !PYTHONPATH=. python3 tools/playtest_simulate.py {c} --engine scene_lm \
      --model-path {OUTPUT_DIR}/{c}/scene_lm_full.pt \
      --n 5 --max-turns 30 --seed 1 \
      --out {OUTPUT_DIR}/{c}/playtest_transcripts.md
print('transcripts on Drive — run the claude_subagent judge sweep locally')


In [ ]:
# ── 10 Playtest transcripts → Drive (judge sweep runs locally) ──────────────
import shutil
for c in CASES:
    shutil.copy(f'{OUTPUT_DIR}/{c}/scene_lm_full.pt', f'outputs/{c}/scene_lm_full.pt') if False else None
    !PYTHONPATH=. python3 tools/playtest_simulate.py {c} --engine scene_lm --n 5 --max-turns 30 --seed 1 \
      --checkpoint {OUTPUT_DIR}/{c}/scene_lm_full.pt 2>/dev/null || \
    PYTHONPATH=. python3 tools/playtest_simulate.py {c} --engine scene_lm --n 5 --max-turns 30 --seed 1
    src = Path('outputs') / c / 'playtest_transcripts.md'
    if src.exists(): shutil.copy(src, Path(OUTPUT_DIR) / c / 'playtest_transcripts.md')
print('transcripts on Drive — run the claude_subagent judge sweep locally')


## After the run

1. Download `scene_lm_full.pt` per case from Drive → `living_tales/trainer/outputs/<case>/` locally.
2. Local: `make eval-gate-scene-lm` (should match), then the judge sweep via `tools/playtest_judge.py` (target ≥18/25; v2 baseline 13/25).
3. Read `error_analysis.md` per case — the error→lever map at the top routes each failing signal to a data fix vs hyperparameter change.
4. If a probe fails: levers in order — more adapter steps (1500→3000), adapter lr 3e-4→1e-4, then data fixes per the confusion tables.
